# STEP 4B: Pavement Features, Pruning Fixes, sPCI & Validation Sensitivity
**Road PCI Intelligence**

This notebook executes on **CPU** to:
1. Extract geometric distress features from **Ground-Truth** masks across train+val splits ($N = 1,486$ images) with **skeleton pruning** (< 12 px spurs removed).
2. Report before vs. after distributions for `n_junctions` and `network_index`.
3. Compute empirical crack width thresholds ($p_{33}, p_{80}$) and reference network factor ($n_{\text{ref}}$).
4. Evaluate feature fidelity on validation predictions via **Spearman rank correlation** (reporting $\rho$ only).
5. Compute simplified PCI (sPCI) from predicted and GT masks, evaluating rank correlation and MAE.
6. Perform 200-draw parameter sensitivity analysis on validation sPCI rankings.
7. Generate and verify blinded human labeling sheet and export summary.

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
import os
import sys
import subprocess
import shutil
import time
import json
from pathlib import Path

print("Installing required dependencies (segmentation-models-pytorch, albumentations, scipy, pyyaml)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8", "scipy==1.17.1", "pyyaml==6.0.2"], check=True)

# Clone repository
repo_dir = Path("/kaggle/working/repo")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

print("Cloning repository from GitHub...")
subprocess.run(["git", "clone", "https://github.com/skilloutofbounds/road-pci-intelligence.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

import torch
import torchvision
import numpy as np
import pandas as pd
import cv2
import yaml
from PIL import Image
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import segmentation_models_pytorch as smp

print("=" * 80)
print("ENVIRONMENT INFORMATION:")
print(f"  • Python:       {sys.version.split()[0]}")
print(f"  • PyTorch:      {torch.__version__}")
print(f"  • Torchvision:  {torchvision.__version__}")
print(f"  • SMP Version:  {smp.__version__}")
print(f"  • CPU Threads:  {torch.get_num_threads()}")
print("=" * 80)

In [ ]:
# ==============================================================================
# 2. RESOLVE DATASET ROOTS & VERIFY FINAL WEIGHTS
# ==============================================================================
import hashlib
from src.build_manifest import DATASET_SPECS, resolve_dataset_root
from src.pavement_features import compute_features, rescale_mask, compute_skeleton_length, prune_skeleton
from src.data_rules import load_standardized_mask
from src.datasets import RoadDistressDataset
from src.spci import compute_spci, load_spci_config, get_condition_class

dataset_roots = {}
for src in ['crack500', 'road_crack', 'pothole']:
    meta = DATASET_SPECS[src]
    resolved = resolve_dataset_root(meta['owner'], meta['name'])
    if resolved is None:
        raise FileNotFoundError(f"CRITICAL: Failed to locate dataset root for {src}")
    dataset_roots[src] = resolved
    print(f"  • {src:<12} -> {resolved}")

weights_candidates = [
    Path("/kaggle/input/datasets/vedshah04/road-pci-seg-final-weights/best.pt"),
    Path("/kaggle/input/road-pci-seg-final-weights/best.pt"),
    repo_dir / "scratch/downloaded_final_weights/best.pt"
]
weights_path = None
for p in weights_candidates:
    if p.exists():
        weights_path = p
        break

if weights_path is None:
    raise FileNotFoundError("CRITICAL: Failed to locate best.pt final weights checkpoint!")

print(f"Found weights at: {weights_path} ({weights_path.stat().st_size:,} bytes)")
sha256 = hashlib.sha256(weights_path.read_bytes()).hexdigest()
print(f"Computed SHA-256: {sha256}")
EXPECTED_SHA = "9269c5edb6e5d37b359bd3337fbd83c60901cd2b241f9ef789e9116fee400ef7"
assert sha256 == EXPECTED_SHA, f"SHA mismatch! Expected {EXPECTED_SHA}, got {sha256}"
print("[PASS] SHA-256 verified strictly.")

In [ ]:
# ==============================================================================
# 3. PHASE 1: COMPUTE GROUND-TRUTH FEATURES ON TRAIN + VAL (N = 1,486)
# ==============================================================================
manifest_path = repo_dir / 'manifests/segmentation_manifest.csv'
manifest_df = pd.read_csv(manifest_path)

trainval_df = manifest_df[manifest_df['split'].isin(['train', 'val'])].copy().reset_index(drop=True)
print(f"Loaded train+val records: {len(trainval_df):,} images")

t0 = time.time()
gt_feature_records = []
all_crack_local_widths = []

for idx, row in trainval_df.iterrows():
    src = row['source']
    split = row['split']
    root = dataset_roots[src]
    mask_rel = Path(row['mask_path_or_label_path'])
    full_mask_path = root / mask_rel
    img_shape = (int(row['height']), int(row['width']))
    
    raw_mask = load_standardized_mask(source=src, mask_or_label_path=full_mask_path, img_shape=img_shape)
    
    if src in ['crack500', 'road_crack']:
        crack_m = raw_mask
        pothole_m = None
    elif src == 'pothole':
        crack_m = None
        pothole_m = raw_mask
    else:
        crack_m = None
        pothole_m = None
        
    # Compute both raw (unpruned) and pruned features
    feats_raw = compute_features(crack_mask=crack_m, pothole_mask=pothole_m, prune_branches=False)
    feats_pruned = compute_features(crack_mask=crack_m, pothole_mask=pothole_m, prune_branches=True, min_branch_len=12)
    
    # Collect local widths for crack-bearing masks
    if src in ['crack500', 'road_crack'] and feats_pruned['crack_length_px'] > 0:
        c_res = rescale_mask(crack_m, 512)
        n_lbl, lbls, stats, _ = cv2.connectedComponentsWithStats(c_res, connectivity=8)
        clean_c = np.zeros_like(c_res)
        for l in range(1, n_lbl):
            if stats[l, cv2.CC_STAT_AREA] >= 30:
                clean_c[lbls == l] = 1
        if np.any(clean_c):
            dt = cv2.distanceTransform(clean_c, cv2.DIST_L2, 5)
            from skimage.morphology import skeletonize
            sk = prune_skeleton(skeletonize(clean_c > 0), min_branch_len=12)
            if np.any(sk):
                widths = 2.0 * dt[sk > 0]
                all_crack_local_widths.extend(widths.tolist())
    
    rec = {
        'source': src,
        'image_path': row['image_path'],
        'split': split,
        'group_id': row['group_id'],
        # Pruned metrics (primary)
        **feats_pruned,
        # Before-pruning metrics for comparison
        'raw_n_junctions': feats_raw['n_junctions'],
        'raw_network_index': feats_raw['network_index'],
    }
    gt_feature_records.append(rec)
    
    if (idx + 1) % 400 == 0 or (idx + 1) == len(trainval_df):
        print(f"  Processed {idx + 1:,} / {len(trainval_df):,} masks in {time.time()-t0:.1f}s")

gt_features_df = pd.DataFrame(gt_feature_records)
print(f"[OK] Extracted features for {len(gt_features_df):,} masks in {time.time()-t0:.2f}s")

# Overwrite manifests/features_gt_trainval.csv
out_csv = repo_dir / "manifests/features_gt_trainval.csv"
gt_features_df.to_csv(out_csv, index=False)
gt_features_df.to_csv(Path("/kaggle/working/features_gt_trainval.csv"), index=False)
print(f"[OK] Saved features to {out_csv}")

In [ ]:
# ==============================================================================
# 4. DISTRIBUTION TABLES & BEFORE VS AFTER PRUNING COMPARISON
# ==============================================================================
print("=" * 105)
print("BEFORE VS. AFTER SKELETON PRUNING (< 12 PX SPURS REMOVED): JUNCTIONS & NETWORK INDEX")
print("=" * 105)
print(f"{'Source':<12} | {'Metric':<20} | {'Raw Median':<14} | {'Pruned Median':<14} | {'Raw P90':<12} | {'Pruned P90':<12}")
print("-" * 105)

for src in ['crack500', 'road_crack', 'pothole']:
    src_df = gt_features_df[gt_features_df['source'] == src]
    
    raw_j_med = np.median(src_df['raw_n_junctions'])
    pruned_j_med = np.median(src_df['n_junctions'])
    raw_j_p90 = np.percentile(src_df['raw_n_junctions'], 90)
    pruned_j_p90 = np.percentile(src_df['n_junctions'], 90)
    
    raw_ni_med = np.median(src_df['raw_network_index'])
    pruned_ni_med = np.median(src_df['network_index'])
    raw_ni_p90 = np.percentile(src_df['raw_network_index'], 90)
    pruned_ni_p90 = np.percentile(src_df['network_index'], 90)
    
    print(f"{src:<12} | {'n_junctions':<20} | {raw_j_med:14.1f} | {pruned_j_med:14.1f} | {raw_j_p90:12.1f} | {pruned_j_p90:12.1f}")
    print(f"{src:<12} | {'network_index':<20} | {raw_ni_med:14.3f} | {pruned_ni_med:14.3f} | {raw_ni_p90:12.3f} | {pruned_ni_p90:12.3f}")
    print("-" * 105)

# Empirical Crack Severity Thresholds
crack_widths_arr = np.array(all_crack_local_widths)
p33_width = float(np.percentile(crack_widths_arr, 33))
p80_width = float(np.percentile(crack_widths_arr, 80))
median_width = float(np.median(crack_widths_arr))

# Empirical n_ref
crack_bearing_df = gt_features_df[(gt_features_df['source'].isin(['crack500', 'road_crack'])) & (gt_features_df['crack_length_px'] > 0)]
n_ref_val = float(np.percentile(crack_bearing_df['network_index'], 90))

print("\n" + "=" * 80)
print("EMPIRICAL CALIBRATION THRESHOLDS (CRACK-BEARING TRAIN+VAL GT MASKS):")
print(f"  • Total Skeleton Pixels Sampled: {len(crack_widths_arr):,}")
print(f"  • Local Width 33rd Percentile (T_low):   {p33_width:.3f} px")
print(f"  • Local Width Median:                    {median_width:.3f} px")
print(f"  • Local Width 80th Percentile (T_high):  {p80_width:.3f} px")
print(f"  • Network Index 90th Percentile (n_ref): {n_ref_val:.3f}")
print("=" * 80)

In [ ]:
# ==============================================================================
# 5. VISUAL INSPECTION: 6 VAL SAMPLES PER SOURCE (IMAGE | MASK | PRUNED SKELETON)
# ==============================================================================
from skimage.morphology import skeletonize

val_df = trainval_df[trainval_df['split'] == 'val'].reset_index(drop=True)

for src in ['crack500', 'road_crack', 'pothole']:
    src_val = val_df[val_df['source'] == src].reset_index(drop=True)
    n_total = len(src_val)
    step = max(1, n_total // 6)
    indices = [min(i * step, n_total - 1) for i in range(6)]
    indices = sorted(list(dict.fromkeys(indices)))
    while len(indices) < 6:
        indices.append(len(indices))
        
    samples = src_val.iloc[indices]
    
    fig, axes = plt.subplots(6, 3, figsize=(15, 24))
    plt.subplots_adjust(top=0.96, bottom=0.03, hspace=0.32, wspace=0.10)
    fig.suptitle(f"Pruned Feature Inspection: {src.upper()} Validation Set (6 Samples)", fontsize=14, fontweight='bold', y=0.98)
    
    for row_idx, (_, row) in enumerate(samples.iterrows()):
        root = dataset_roots[src]
        img_full = root / row['image_path']
        mask_full = root / row['mask_path_or_label_path']
        img_shape = (int(row['height']), int(row['width']))
        
        with Image.open(img_full) as pil_img:
            img_rgb = np.array(pil_img.convert('RGB'))
            
        raw_mask = load_standardized_mask(src, mask_full, img_shape=img_shape)
        mask_512 = rescale_mask(raw_mask, target_shorter_side=512)
        h5, w5 = mask_512.shape[:2]
        img_512 = cv2.resize(img_rgb, (w5, h5), interpolation=cv2.INTER_LINEAR)
        
        stem = Path(row['image_path']).name
        if len(stem) > 24:
            stem = stem[:21] + "..."
            
        axes[row_idx, 0].imshow(img_512)
        axes[row_idx, 0].set_title(f"[Sample {row_idx+1}] {stem}\nImage ({w5}x{h5})", fontsize=8.5, fontweight='bold')
        axes[row_idx, 0].axis('off')
        
        mask_overlay = img_512.copy()
        color = [230, 30, 30] if src != 'pothole' else [30, 110, 240]
        fg = (mask_512 > 0)
        mask_overlay[fg] = (mask_overlay[fg].astype(float) * 0.45 + np.array(color) * 0.55).astype(np.uint8)
        axes[row_idx, 1].imshow(mask_overlay)
        axes[row_idx, 1].set_title(f"GT Mask ({'crack' if src != 'pothole' else 'pothole'})\nPixels: {int(fg.sum()):,} ({fg.mean()*100:.2f}%)", fontsize=8.5)
        axes[row_idx, 1].axis('off')
        
        skel_canvas = img_512.copy()
        if src != 'pothole':
            n_lbl, lbls, stats, _ = cv2.connectedComponentsWithStats(mask_512, connectivity=8)
            clean_crack = np.zeros_like(mask_512)
            for l in range(1, n_lbl):
                if stats[l, cv2.CC_STAT_AREA] >= 30:
                    clean_crack[lbls == l] = 1
            skel_raw = skeletonize(clean_crack > 0)
            skel = prune_skeleton(skel_raw, min_branch_len=12)
            skel_canvas[skel] = [0, 255, 0]
            
            sk_u8 = skel.astype(np.uint8)
            k_n = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], dtype=np.uint8)
            nc = cv2.filter2D(sk_u8, -1, k_n) * sk_u8
            junc_pts = ((nc >= 3) & (sk_u8 > 0))
            jy, jx = np.where(junc_pts)
            for y_j, x_j in zip(jy, jx):
                cv2.circle(skel_canvas, (x_j, y_j), 4, (255, 0, 0), thickness=-1)
                
            length_val = compute_skeleton_length(skel)
            num_j_lbl, _ = cv2.connectedComponents(junc_pts.astype(np.uint8), connectivity=8)
            n_j = max(0, num_j_lbl - 1)
            axes[row_idx, 2].imshow(skel_canvas)
            axes[row_idx, 2].set_title(f"Pruned Skeleton & Junctions\nLength: {length_val:.1f} px | Junctions: {n_j}", fontsize=8.5, fontweight='bold')
        else:
            n_lbl, lbls, stats, _ = cv2.connectedComponentsWithStats(mask_512, connectivity=8)
            p_cnt = 0
            for l in range(1, n_lbl):
                if stats[l, cv2.CC_STAT_AREA] >= 50:
                    p_cnt += 1
                    comp_mask = (lbls == l).astype(np.uint8)
                    contours, _ = cv2.findContours(comp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(skel_canvas, contours, -1, (255, 255, 0), 2)
            axes[row_idx, 2].imshow(skel_canvas)
            axes[row_idx, 2].set_title(f"Pothole Contours (Yellow)\nValid Potholes: {p_cnt}", fontsize=8.5, fontweight='bold')
        axes[row_idx, 2].axis('off')
        
    out_fig = Path(f"/kaggle/working/val_feature_skeletons_{src}.png")
    plt.savefig(out_fig, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"[OK] Saved figure: {out_fig}")

In [ ]:
# ==============================================================================
# 6. PHASE 3: VALIDATION PREDICTIONS, SPEARMAN RHO TABLE & sPCI EVALUATION
# ==============================================================================
device = torch.device('cpu')
model = smp.Unet(encoder_name='resnet34', classes=2)
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

val_dataset = RoadDistressDataset(manifest_df=manifest_df, dataset_roots=dataset_roots, split='val', crop_size=512)
print(f"Validation DataLoader: {len(val_dataset)} images")

t0_eval = time.time()
paired_val_records = []

# Load SPCI config with empirical thresholds
spci_cfg = load_spci_config(repo_dir / "configs/spci_params.yaml")
spci_cfg["crack_severity"]["low_threshold_px"] = p33_width
spci_cfg["crack_severity"]["high_threshold_px"] = p80_width
spci_cfg["network_factor"]["n_ref"] = n_ref_val

with torch.no_grad():
    for idx in range(len(val_dataset)):
        image_t, target_t, label_mask, metas = val_dataset[idx]
        image_b = image_t.unsqueeze(0)
        src = metas['source']
        unpad_info = metas['eval_unpad']
        y0, x0, h, w = unpad_info
        
        # Forward Flip-TTA
        probs_orig = torch.sigmoid(model(image_b))
        probs_h = torch.flip(torch.sigmoid(model(torch.flip(image_b, dims=[-1]))), dims=[-1])
        probs_v = torch.flip(torch.sigmoid(model(torch.flip(image_b, dims=[-2]))), dims=[-2])
        probs_tta = (probs_orig + probs_h + probs_v) / 3.0
        
        p_crack = (probs_tta[0, 0, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8)
        p_pot = (probs_tta[0, 1, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8)
        
        t_crack = (target_t[0, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8) if label_mask[0] else None
        t_pot = (target_t[1, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8) if label_mask[1] else None
        
        # Compute pruned features
        gt_feats = compute_features(t_crack, t_pot, prune_branches=True, min_branch_len=12)
        pred_feats = compute_features(p_crack, p_pot, prune_branches=True, min_branch_len=12)
        
        # Compute sPCI
        spci_pred = compute_spci(p_crack, p_pot, config=spci_cfg)
        spci_gt = compute_spci(t_crack, t_pot, config=spci_cfg)
        
        paired_val_records.append({
            'source': src,
            'image_path': metas['image_path'],
            'gt': gt_feats,
            'pred': pred_feats,
            'spci_gt': spci_gt['spci'],
            'spci_pred': spci_pred['spci'],
            'pred_p_crack': p_crack,
            'pred_p_pot': p_pot,
            'gt_t_crack': t_crack,
            'gt_t_pot': t_pot,
        })
        
        if (idx + 1) % 50 == 0 or (idx + 1) == len(val_dataset):
            print(f"  Evaluated {idx + 1} / {len(val_dataset)} images ({((idx+1)/len(val_dataset))*100:.1f}%) in {time.time()-t0_eval:.1f}s")

print(f"[OK] Completed validation evaluation in {time.time()-t0_eval:.2f}s")

# Spearman Rank Correlation Table (Rho only)
eval_targets = {
    'crack500': ['crack_length_px', 'relative_width', 'network_index'],
    'road_crack': ['crack_length_px', 'relative_width', 'network_index'],
    'pothole': ['pothole_area_frac']
}

print("\n" + "=" * 70)
print("SPEARMAN RANK CORRELATION: GT vs. PREDICTED FEATURES (VALIDATION SPLIT)")
print("=" * 70)
print(f"{'Source':<12} | {'N Images':<10} | {'Feature':<22} | {'Spearman rho':<14}")
print("-" * 70)

feat_corr_results = {}
for src, feat_list in eval_targets.items():
    src_recs = [r for r in paired_val_records if r['source'] == src]
    n_img = len(src_recs)
    feat_corr_results[src] = {}
    for feat in feat_list:
        gt_vals = np.array([r['gt'][feat] for r in src_recs])
        pred_vals = np.array([r['pred'][feat] for r in src_recs])
        if np.std(gt_vals) == 0 or np.std(pred_vals) == 0:
            rho = float('nan')
        else:
            rho = float(spearmanr(gt_vals, pred_vals).statistic)
        feat_corr_results[src][feat] = rho
        print(f"{src:<12} | {n_img:<10} | {feat:<22} | {rho:14.4f}")
    print("-" * 70)
print("=" * 70)

# sPCI GT vs Predicted Comparison
print("\n" + "=" * 70)
print("sPCI VALIDATION COMPARISON: GROUND-TRUTH vs. PREDICTED MASKS")
print("=" * 70)
print(f"{'Source':<12} | {'N Images':<10} | {'Spearman rho':<14} | {'MAE (Points)':<14}")
print("-" * 70)

spci_comparison_results = {}
for src in ['crack500', 'road_crack', 'pothole']:
    src_recs = [r for r in paired_val_records if r['source'] == src]
    n_img = len(src_recs)
    gt_spci = np.array([r['spci_gt'] for r in src_recs])
    pred_spci = np.array([r['spci_pred'] for r in src_recs])
    
    rho = float(spearmanr(gt_spci, pred_spci).statistic)
    mae = float(np.mean(np.abs(gt_spci - pred_spci)))
    spci_comparison_results[src] = {'n': n_img, 'rho': rho, 'mae': mae}
    print(f"{src:<12} | {n_img:<10} | {rho:14.4f} | {mae:14.2f}")
print("=" * 70)

In [ ]:
# ==============================================================================
# 7. PHASE 4: sPCI SENSITIVITY ANALYSIS (200 DRAWS ON VAL PREDICTIONS)
# ==============================================================================
import copy

print("\nRunning 200-draw sensitivity analysis on validation predicted masks...")
rng = np.random.RandomState(42)

# Pre-extract validation predictions
val_preds = [(r['pred_p_crack'], r['pred_p_pot']) for r in paired_val_records]
baseline_spci_vals = np.array([r['spci_pred'] for r in paired_val_records])

# 1. Perturb deduct curve parameters (a, b) in [0.7, 1.3]
rhos_curves = []
param_rhos = {}

for curve_key in spci_cfg["deduct_curves"].keys():
    param_rhos[curve_key] = []

for draw in range(200):
    cfg_p = copy.deepcopy(spci_cfg)
    for curve_name, p_dict in cfg_p["deduct_curves"].items():
        factor_a = rng.uniform(0.7, 1.3)
        factor_b = rng.uniform(0.7, 1.3)
        p_dict["a"] *= factor_a
        p_dict["b"] *= factor_b
        
    draw_spci = np.array([compute_spci(pc, pp, config=cfg_p)['spci'] for pc, pp in val_preds])
    rho = float(spearmanr(baseline_spci_vals, draw_spci).statistic)
    rhos_curves.append(rho)

# 2. Perturb severity thresholds and beta in [0.7, 1.3]
rhos_thresholds_beta = []
for draw in range(200):
    cfg_p = copy.deepcopy(spci_cfg)
    cfg_p["crack_severity"]["low_threshold_px"] *= rng.uniform(0.7, 1.3)
    cfg_p["crack_severity"]["high_threshold_px"] *= rng.uniform(0.7, 1.3)
    cfg_p["network_factor"]["beta"] *= rng.uniform(0.7, 1.3)
    
    draw_spci = np.array([compute_spci(pc, pp, config=cfg_p)['spci'] for pc, pp in val_preds])
    rho = float(spearmanr(baseline_spci_vals, draw_spci).statistic)
    rhos_thresholds_beta.append(rho)

# 3. Individual parameter impact analysis (50 draws each)
individual_sensitivity = {}
for p_group in ['crack_deduct_a', 'pothole_deduct_a', 'severity_thresholds', 'network_beta']:
    rhos_group = []
    for _ in range(50):
        cfg_p = copy.deepcopy(spci_cfg)
        if p_group == 'crack_deduct_a':
            for k in ['crack_low', 'crack_medium', 'crack_high']:
                cfg_p["deduct_curves"][k]["a"] *= rng.uniform(0.7, 1.3)
        elif p_group == 'pothole_deduct_a':
            for k in ['pothole_low', 'pothole_medium', 'pothole_high']:
                cfg_p["deduct_curves"][k]["a"] *= rng.uniform(0.7, 1.3)
        elif p_group == 'severity_thresholds':
            cfg_p["crack_severity"]["low_threshold_px"] *= rng.uniform(0.7, 1.3)
            cfg_p["crack_severity"]["high_threshold_px"] *= rng.uniform(0.7, 1.3)
        elif p_group == 'network_beta':
            cfg_p["network_factor"]["beta"] *= rng.uniform(0.7, 1.3)
            
        draw_spci = np.array([compute_spci(pc, pp, config=cfg_p)['spci'] for pc, pp in val_preds])
        rhos_group.append(float(spearmanr(baseline_spci_vals, draw_spci).statistic))
    individual_sensitivity[p_group] = {
        'min': float(np.min(rhos_group)),
        'p5': float(np.percentile(rhos_group, 5)),
        'median': float(np.median(rhos_group))
    }

print("=" * 80)
print("sPCI RANKING SENSITIVITY ANALYSIS (200 DRAWS, PERTURBATION IN [0.7, 1.3]):")
print("=" * 80)
print(f"Deduct Curves (a, b)   : Min rho = {np.min(rhos_curves):.4f} | P5 rho = {np.percentile(rhos_curves, 5):.4f} | Median rho = {np.median(rhos_curves):.4f}")
print(f"Thresholds & Beta      : Min rho = {np.min(rhos_thresholds_beta):.4f} | P5 rho = {np.percentile(rhos_thresholds_beta, 5):.4f} | Median rho = {np.median(rhos_thresholds_beta):.4f}")
print("-" * 80)
print("Individual Parameter Group Robustness (50 draws each):")
for grp, res in individual_sensitivity.items():
    print(f"  • {grp:<22}: Min rho = {res['min']:.4f} | P5 rho = {res['p5']:.4f} | Median rho = {res['median']:.4f}")

most_sensitive = min(individual_sensitivity.keys(), key=lambda k: individual_sensitivity[k]['min'])
print(f"\nMost Sensitive Parameter Group: {most_sensitive} (Lowest min ranking correlation: {individual_sensitivity[most_sensitive]['min']:.4f})")
print("=" * 80)

In [ ]:
# ==============================================================================
# 8. HUMAN LABELING SHEET VERIFICATION & SUMMARY EXPORT
# ==============================================================================
import json

summary_payload = {
    'weights_sha256': sha256,
    'trainval_records': len(gt_features_df),
    'calibration': {
        'p33_width_px': p33_width,
        'median_width_px': median_width,
        'p80_width_px': p80_width,
        'n_ref': n_ref_val
    },
    'feat_spearman_rho': feat_corr_results,
    'spci_val_comparison': spci_comparison_results,
    'sensitivity': {
        'curves_a_b': {
            'min': float(np.min(rhos_curves)),
            'p5': float(np.percentile(rhos_curves, 5)),
            'median': float(np.median(rhos_curves))
        },
        'thresholds_beta': {
            'min': float(np.min(rhos_thresholds_beta)),
            'p5': float(np.percentile(rhos_thresholds_beta, 5)),
            'median': float(np.median(rhos_thresholds_beta))
        },
        'individual': individual_sensitivity,
        'most_sensitive': most_sensitive
    },
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}

with open("/kaggle/working/step_4b_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)
print("[OK] Exported /kaggle/working/step_4b_summary.json successfully.")

# Copy updated spci_params.yaml with empirical values to repo configs
spci_cfg_export = copy.deepcopy(spci_cfg)
with open(repo_dir / "configs/spci_params.yaml", "w", encoding="utf-8") as f:
    yaml.dump(spci_cfg_export, f, sort_keys=False)
print("[OK] Synchronized configs/spci_params.yaml with empirical thresholds.")